# Movie Sentiment Classification

## Problem Statement
A movie review platform wants to analyze the sentiments expressed in user reviews. The review text is available in the `phrase` column, and it has also been used to derive three additional features: `feature_1`, `feature_2`, and `feature_3`.

This is a **multi-class sentiment analysis** problem where the goal is to predict the sentiment label for each review:
- `0` -> Negative
- `1` -> Neutral
- `2` -> Positive

## Dataset Description
The competition provides the following files:
- `train.csv` - training data with input features and the target column
- `test.csv` - test data where the `sentiment` column is hidden
- `sample_submission.csv` - sample file showing the required submission format

The dataset contains these columns:
- `id`: unique row identifier
- `phrase`: movie review text
- `feature_1`, `feature_2`, `feature_3`: engineered features created from the review text
- `sentiment`: target label available only in the training set

## Workflow
In this notebook, we will:
- inspect the data and understand its structure
- perform exploratory data analysis on both text-derived and numeric features
- clean the dataset by checking duplicates and missing values
- combine TF-IDF text features with the engineered numeric features
- train multiple classification models and compare their performance
- use stacking to improve the final prediction and create a Kaggle submission


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns

import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from scipy.sparse import csr_matrix,hstack

# Supress Warnings

import warnings
warnings.filterwarnings('ignore')


## Data Loading

We load the three competition files: `train.csv`, `test.csv`, and `sample_submission.csv`. The training data contains both features and the target label, while the test data contains only features and is used for final prediction.


In [ ]:
train=pd.read_csv("/kaggle/input/competitions/mlp-jan-26-2025-kaggle-assignment-3/train.csv")
test=pd.read_csv("/kaggle/input/competitions/mlp-jan-26-2025-kaggle-assignment-3/test.csv")

sample=pd.read_csv('/kaggle/input/competitions/mlp-jan-26-2025-kaggle-assignment-3/sample_submission.csv')


In [ ]:
train.head()

## Data Types

Inspecting data types helps us separate the raw text field (`phrase`) from the numeric engineered features. This is important because text and numeric columns require different preprocessing strategies before model training.


In [ ]:
train.info()

## Descriptive Statistics

Here we summarize the numerical columns to understand their scale, spread, and central tendency. Statistics such as mean, standard deviation, and min/max values help us identify whether the engineered features have reasonable ranges before modeling.


In [ ]:
train.describe()

## Exploratory Data Analysis (EDA)

In the EDA section, we examine the target distribution and study how the engineered features behave across sentiment classes. This helps us understand whether the numeric features add useful signal beyond the original review text.


In [ ]:
sns.countplot(x='sentiment', data=train)
plt.title("Sentiment Distribution")
plt.show()

### Insight:
The count plot shows how the three sentiment classes are distributed in the training set. If the classes are reasonably balanced, accuracy becomes a more reliable evaluation metric; if not, we must be careful about class imbalance during training and validation.


In [ ]:
train[['feature_1','feature_2','feature_3']].hist(figsize=(10,5))
plt.show()

### Insight:
The histograms help us understand the spread and shape of `feature_1`, `feature_2`, and `feature_3`. Since these features were derived from the review text, their distributions may capture useful sentiment-related patterns.


In [ ]:
sns.boxplot(x='sentiment', y='feature_1', data=train)
plt.show()

sns.boxplot(x='sentiment', y='feature_2', data=train)
plt.show()

sns.boxplot(x='sentiment', y='feature_3', data=train)
plt.show()

### Insight:
The boxplots allow us to compare each engineered feature across negative, neutral, and positive reviews. Clear shifts in the distributions suggest that these features may be informative for the sentiment classification task.


## Duplicate Handling

We check whether the training data contains duplicate rows. Duplicates can bias model performance, especially in text classification problems, because the same review may appear multiple times and artificially inflate validation scores.


In [ ]:
train.duplicated().sum()

### Insight:
Based on the output, we can decide whether duplicate rows need to be removed. If no duplicates are present, we can move forward without changing the dataset.


## Outlier Detection

Although this is primarily a text classification problem, the engineered numeric features can still contain extreme values. We use the IQR method to inspect potential outliers and understand whether any rows look unusual.


In [ ]:
num_cols = train.select_dtypes(include=['int64', 'float64']).columns

def detect_outliers_iqr(train, col):
    Q1 = train[col].quantile(0.25)
    Q3 = train[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = train[(train[col] < lower_bound) | (train[col] > upper_bound)]
    return outliers

for col in num_cols:
    outliers = detect_outliers_iqr(train, col)
    print(f"{col}: {len(outliers)} outliers")


### Insight:
Potential outliers are examined as part of data understanding, but they are not removed automatically. This is a reasonable choice here because several of the planned models, especially tree-based methods, are fairly robust to outlier values.


## Missing Value Handling

Before training, we check for missing values and handle them consistently in both the training and test sets. The numeric engineered features are imputed using their column means so that the models can train without errors.


In [ ]:
train.isna().sum()


In [ ]:
def fill(df):
    df=df.copy()
    df['feature_1']=df['feature_1'].fillna(df['feature_1'].mean())
    df['feature_2']=df['feature_2'].fillna(df['feature_2'].mean())
    df['feature_3']=df['feature_3'].fillna(df['feature_3'].mean())
    return df


In [ ]:
train=fill(train)
test=fill(test)


In [ ]:
X=train.drop(columns=['sentiment','id'])
y=train['sentiment']

test=test.drop(columns=['id'])


In [ ]:
lgb_model = Pipeline([
    ('model', LGBMClassifier(random_state=42, verbosity=-1,n_jobs=-1))
])

xgb_model = Pipeline([
    ('model', XGBClassifier(random_state=42, verbosity=0,n_jobs=-1))
])

cat_model = Pipeline([
    ('model', CatBoostClassifier(random_state=42, verbose=False))
])

rf_model = Pipeline([
    ('model', RandomForestClassifier(n_jobs=-1,random_state=42))
])

et_model = Pipeline([
    ('model', ExtraTreesClassifier(n_jobs=-1,random_state=42))
])

gb_model = Pipeline([
    ('model', GradientBoostingClassifier(max_depth=3,random_state=42))
])

lr_model = Pipeline([
    ('model', LogisticRegression(max_iter=1000,n_jobs=-1))
])


## Feature Processing

The `phrase` column is the main source of sentiment information, so it is converted into TF-IDF representations at both word and character levels. These sparse text features are then combined with `feature_1`, `feature_2`, and `feature_3` so that the models can learn from both the raw review text and the derived numeric signals.

Feature strategy used in this notebook:
- TF-IDF on the review text to capture important words and character patterns
- direct use of the three engineered numeric features
- no additional scaling for tree-based models, which generally do not require it


## Model Building

To solve this multi-class sentiment classification problem, we train a diverse set of models. The goal is to compare linear, bagging, boosting, and gradient-based approaches on the same feature representation.

Models trained in this notebook:
- LightGBM
- XGBoost
- CatBoost
- RandomForest
- ExtraTrees
- GradientBoosting
- LogisticRegression


## Cross Validation

We use **Stratified K-Fold cross-validation** with 5 splits so that each fold preserves the class distribution of the target variable. This gives a more reliable estimate of model performance on a three-class sentiment prediction task.


## Model Comparison

Each model is evaluated using cross-validation accuracy. Comparing the models on the same folds helps us identify which approaches capture sentiment patterns most effectively before building the final ensemble.


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

n_classes = 3

oof_lgb = np.zeros((len(X), n_classes))
oof_xgb = np.zeros((len(X), n_classes))
oof_cat = np.zeros((len(X), n_classes))
oof_rf  = np.zeros((len(X), n_classes))
oof_et  = np.zeros((len(X), n_classes))
oof_gb  = np.zeros((len(X), n_classes))
oof_lr  = np.zeros((len(X), n_classes))

test_lgb = np.zeros((len(test), n_classes))
test_xgb = np.zeros((len(test), n_classes))
test_cat = np.zeros((len(test), n_classes))
test_rf  = np.zeros((len(test), n_classes))
test_et  = np.zeros((len(test), n_classes))
test_gb  = np.zeros((len(test), n_classes))
test_lr  = np.zeros((len(test), n_classes))

oof_preds = np.zeros(len(X))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    
    print(f"\n===== Fold {fold+1} =====")
    
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    tfidf_word = TfidfVectorizer(
        max_features=10000,
        analyzer='word',
        ngram_range=(1,2),
        sublinear_tf=True,
        strip_accents='unicode'
    )
    
    tfidf_char = TfidfVectorizer(
        max_features=5000,
        analyzer='char_wb',
        ngram_range=(3,4),
        sublinear_tf=True,
        strip_accents='unicode'
    )
    
    X_tr_word = tfidf_word.fit_transform(X_tr['phrase'])
    X_val_word = tfidf_word.transform(X_val['phrase'])
    test_word = tfidf_word.transform(test['phrase'])
    
    X_tr_char = tfidf_char.fit_transform(X_tr['phrase'])
    X_val_char = tfidf_char.transform(X_val['phrase'])
    test_char = tfidf_char.transform(test['phrase'])
    
    X_tr_num = csr_matrix(X_tr[['feature_1','feature_2','feature_3']].values)
    X_val_num = csr_matrix(X_val[['feature_1','feature_2','feature_3']].values)
    test_num = csr_matrix(test[['feature_1','feature_2','feature_3']].values)
    
    X_tr_final = hstack([X_tr_word, X_tr_char, X_tr_num])
    X_val_final = hstack([X_val_word, X_val_char, X_val_num])
    test_final = hstack([test_word, test_char, test_num])
    
    lgb_model.fit(X_tr_final, y_tr)
    xgb_model.fit(X_tr_final, y_tr)
    cat_model.fit(X_tr_final, y_tr)
    rf_model.fit(X_tr_final, y_tr)
    et_model.fit(X_tr_final, y_tr)
    gb_model.fit(X_tr_final, y_tr)
    lr_model.fit(X_tr_final, y_tr)
    
    oof_lgb[val_idx] = lgb_model.predict_proba(X_val_final)
    oof_xgb[val_idx] = xgb_model.predict_proba(X_val_final)
    oof_cat[val_idx] = cat_model.predict_proba(X_val_final)
    oof_rf[val_idx]  = rf_model.predict_proba(X_val_final)
    oof_et[val_idx]  = et_model.predict_proba(X_val_final)
    oof_gb[val_idx]  = gb_model.predict_proba(X_val_final)
    oof_lr[val_idx]  = lr_model.predict_proba(X_val_final)
    
    test_lgb += lgb_model.predict_proba(test_final) / skf.n_splits
    test_xgb += xgb_model.predict_proba(test_final) / skf.n_splits
    test_cat += cat_model.predict_proba(test_final) / skf.n_splits
    test_rf  += rf_model.predict_proba(test_final) / skf.n_splits
    test_et  += et_model.predict_proba(test_final) / skf.n_splits
    test_gb  += gb_model.predict_proba(test_final) / skf.n_splits
    test_lr  += lr_model.predict_proba(test_final) / skf.n_splits
    
    def get_acc(oof):
        return accuracy_score(y_val, np.argmax(oof[val_idx], axis=1))
    
    print("LGB:", get_acc(oof_lgb))
    print("XGB:", get_acc(oof_xgb))
    print("CAT:", get_acc(oof_cat))
    print("RF :", get_acc(oof_rf))
    print("ET :", get_acc(oof_et))
    print("GB :", get_acc(oof_gb))
    print("LR :", get_acc(oof_lr))
    
    oof_avg = (
        oof_lgb[val_idx] +
        oof_xgb[val_idx] +
        oof_cat[val_idx] +
        oof_rf[val_idx]  +
        oof_et[val_idx]  +
        oof_gb[val_idx]  +
        oof_lr[val_idx]
    ) / 7
    
    fold_preds = np.argmax(oof_avg, axis=1)
    oof_preds[val_idx] = fold_preds
    
    fold_acc = accuracy_score(y_val, fold_preds)
    print("STACK:", fold_acc)

print("\n===== FINAL CV SCORES =====")

def final_acc(oof):
    return accuracy_score(y, np.argmax(oof, axis=1))

print("LGB :", final_acc(oof_lgb))
print("XGB :", final_acc(oof_xgb))
print("CAT :", final_acc(oof_cat))
print("RF  :", final_acc(oof_rf))
print("ET  :", final_acc(oof_et))
print("GB  :", final_acc(oof_gb))
print("LR  :", final_acc(oof_lr))

overall_acc = accuracy_score(y, oof_preds)
print("\nSTACKED FINAL:", overall_acc)


### Insight:
The comparison stage helps reveal whether the engineered features meaningfully support the text representation and which individual models are strongest. The out-of-fold predictions from these base models are then used as inputs for stacking, which can improve the final leaderboard performance.


## Model Stacking

Instead of relying on a single classifier, we combine the out-of-fold prediction probabilities from all base models and train a meta-model on top of them. This stacking approach allows the final model to learn how to best combine the strengths of the individual classifiers.


## Final Submission

After fitting the stacking model, we generate predictions for the hidden test set and write them to `submission.csv` in the required Kaggle format. This file can then be uploaded to the competition for evaluation.


In [ ]:
X_meta = np.hstack([oof_lgb, oof_xgb, oof_cat,oof_rf, oof_et, oof_gb, oof_lr])
test_meta = np.hstack([test_lgb, test_xgb, test_cat,test_rf, test_et, test_gb, test_lr])

meta_model = LGBMClassifier(random_state=42,n_jobs=-1)
meta_model.fit(X_meta, y)

final_preds = meta_model.predict(test_meta)

sample['sentiment']=final_preds

sample.to_csv('submission.csv', index=False)
